In [ ]:
# 1. Kết nối với Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)


Mounted at /content/gdrive


In [ ]:
# 1. CÀI ĐẶT MÔI TRƯỜNG VÀ THƯ VIỆN (Chỉ chạy 1 lần)
# Thêm -qq và -q để ẩn bớt log cài đặt cho gọn màn hình
!apt-get install openjdk-11-jdk-headless -qq -y
!pip install pyspark==3.5.0 findspark notebook plotly pandas matplotlib graphviz -q

# Ghi nhận Jupyter kernel
!python -m ipykernel install --user --name pyspark-env --display-name "Python (PySpark)"

# 2. IMPORT THƯ VIỆN & THIẾT LẬP BIẾN MÔI TRƯỜNG
import os
import sys
import findspark
import pyspark
import pandas as pd
import matplotlib

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.12/dist-packages/pyspark"

# Khởi tạo findspark
findspark.init()

# 3. KIỂM TRA VÀ IN VERSION (Theo yêu cầu)
print("="*30)
print("THÔNG TIN MÔI TRƯỜNG & PHIÊN BẢN")
print("="*30)
print(f"Python version: {sys.version.split(' ')[0]}")
!java -version
print(f"PySpark version: {pyspark.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"JAVA_HOME = {os.environ.get('JAVA_HOME')}")
print(f"SPARK_HOME = {os.environ.get('SPARK_HOME')}")
print("="*30)

# 4. KHỞI TẠO SPARK SESSION & TEST
from pyspark.sql import SparkSession

# Gộp cấu hình offHeap vào SparkSession chính thức
spark = SparkSession.builder \
    .appName("Olist_Machine_Learning") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "10g") \
    .getOrCreate()

print("\nĐã khởi tạo SparkSession thành công!")

# Chạy test DataFrame để đảm bảo Spark đang hoạt động tốt
df = spark.createDataFrame([(1, "hello"), (2, "world")], ["id", "text"])
df.show()

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Installed kernelspec pyspark-env in /root/.local/share/jupyter/kernels/pyspark-env
THÔNG TIN MÔI TRƯỜNG & PHIÊN BẢN
Python version: 3.12.13
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
PySpark version: 4.0.2
Pandas version: 2.2.2
Matplotlib version: 3.10.0
JAVA_HOME = /usr/lib/jvm/java-11-openjdk-amd64
SPARK_HOME = /usr/local/lib/python3.12/dist-packages/pyspark


TypeError: 'JavaPackage' object is not callable

In [ ]:
data_path = "/content/gdrive/MyDrive/BIGDATA/Dataset cuối kỳ/Dataset cuối kỳ/cleaned_master_data.parquet"

# Lúc này biến 'spark' đã tồn tại nên sẽ không bị lỗi nữa
df_master = spark.read.parquet(data_path)

# Kiểm tra xem dữ liệu đã lên chưa
df_master.show(5)

In [ ]:
from pyspark.sql import functions as F

print("[+] BẮT ĐẦU XỬ LÝ TỌA ĐỘ VÀ TÍNH KHOẢNG CÁCH...")

# 1. Đọc file geolocation
# (Thay đổi đường dẫn nếu file csv của bạn nằm ở thư mục khác trên Drive)
df_geo = spark.read.csv("/content/gdrive/MyDrive/BIGDATA/Dataset cuối kỳ/Dataset cuối kỳ/olist_geolocation_dataset.csv", header=True, inferSchema=True)

# 2. Xử lý file Geolocation (RẤT QUAN TRỌNG)
# Trong Olist, 1 mã zip code có thể có nhiều tọa độ do được ghi nhận nhiều lần.
# Phải tính trung bình (avg) để tránh làm duplicate dữ liệu khi Join.
df_geo_unique = df_geo.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("lat"),
    F.avg("geolocation_lng").alias("lng")
)

# 3. Đổi tên cột tọa độ Customer hiện tại cho rõ ràng
df_master = df_master.withColumnRenamed("geolocation_lat", "customer_lat") \
                     .withColumnRenamed("geolocation_lng", "customer_lng")

# 4. JOIN dữ liệu để lấy tọa độ của Seller
# Đổi kiểu dữ liệu zip code của df_geo_unique cho khớp với df_master nếu cần
df_master = df_master.join(
    df_geo_unique,
    df_master["seller_zip_code_prefix"] == df_geo_unique["geolocation_zip_code_prefix"],
    "left"
).withColumnRenamed("lat", "seller_lat") \
 .withColumnRenamed("lng", "seller_lng") \
 .drop("geolocation_zip_code_prefix") # Xóa cột thừa sau khi join

# 5. TÍNH KHOẢNG CÁCH HAVERSINE (KILOMET)
# Chuyển đổi tọa độ sang radians và áp dụng công thức vật lý
df_master = df_master.withColumn("lat1_rad", F.radians(F.col("customer_lat"))) \
                     .withColumn("lon1_rad", F.radians(F.col("customer_lng"))) \
                     .withColumn("lat2_rad", F.radians(F.col("seller_lat"))) \
                     .withColumn("lon2_rad", F.radians(F.col("seller_lng"))) \
                     .withColumn("dlon", F.col("lon2_rad") - F.col("lon1_rad")) \
                     .withColumn("dlat", F.col("lat2_rad") - F.col("lat1_rad")) \
                     .withColumn("a", F.pow(F.sin(F.col("dlat") / 2), 2) + F.cos(F.col("lat1_rad")) * F.cos(F.col("lat2_rad")) * F.pow(F.sin(F.col("dlon") / 2), 2)) \
                     .withColumn("distance_km", 2 * F.atan2(F.sqrt(F.col("a")), F.sqrt(1 - F.col("a"))) * 6371)

# Dọn dẹp các cột tính toán trung gian
df_master = df_master.drop("lat1_rad", "lon1_rad", "lat2_rad", "lon2_rad", "dlon", "dlat", "a")

# Kiểm tra thành quả
df_master.select("customer_zip_code_prefix", "seller_zip_code_prefix", "customer_lat", "seller_lat", "distance_km").show(5)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (SQLTransformer, StringIndexer, OneHotEncoder,
                                VectorAssembler, StandardScaler, UnivariateFeatureSelector)
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

print("================ 1. CHUẨN BỊ FEATURE SET & XỬ LÝ OUTLIER ================")
target_col = "freight_value"

# Lọc bỏ Null ở các cột gốc cần thiết (Bao gồm cả distance_km vừa tính ở bước trước)
raw_cols = ["price", "product_length_cm", "product_height_cm", "product_width_cm",
            "product_weight_g", "customer_state", "seller_state", "distance_km", target_col]
df_model = df_master.select(raw_cols).dropna()

# Xử lý Outlier bằng IQR cho freight_value
Q1 = df_model.approxQuantile(target_col, [0.25], 0.01)[0]
Q3 = df_model.approxQuantile(target_col, [0.75], 0.01)[0]
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

df_model = df_model.filter(F.col(target_col) <= upper_bound)
print(f"-> Ngưỡng chặn Outlier Phí vận chuyển: <= {upper_bound:.2f} BRL")
print(f"-> Số dòng dữ liệu sau khi làm sạch: {df_model.count()}")

print("\n================ 2. XÂY DỰNG BASE PIPELINE (TÍCH HỢP LOGISTICS ĐỂ ĐƯA LÊN WEB) ================")
# SQLTransformer: Tính thể tích, khối lượng quy đổi và check giao hàng liên tỉnh
sql_trans = SQLTransformer(statement="""
    SELECT *,
           (product_length_cm * product_height_cm * product_width_cm) AS product_volume_cm3,
           ((product_length_cm * product_height_cm * product_width_cm) / 6000) * 1000 AS volumetric_weight_g,
           CASE WHEN product_weight_g > (((product_length_cm * product_height_cm * product_width_cm) / 6000) * 1000)
                THEN product_weight_g
                ELSE (((product_length_cm * product_height_cm * product_width_cm) / 6000) * 1000)
           END AS chargeable_weight_g,
           CASE WHEN customer_state != seller_state THEN 1.0 ELSE 0.0 END AS is_cross_state
    FROM __THIS__
""")

categorical_cols = ["customer_state", "seller_state"]
indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_vec") for c in categorical_cols]

numeric_cols = ["price", "chargeable_weight_g", "product_volume_cm3", "is_cross_state", "distance_km"]
assembler = VectorAssembler(
    inputCols=numeric_cols + [c + "_vec" for c in categorical_cols],
    outputCol="unscaled_features", handleInvalid="skip"
)

# Chuẩn hóa
scaler = StandardScaler(inputCol="unscaled_features", outputCol="scaled_features", withStd=True, withMean=False)

# CHỌN LỌC ĐẶC TRƯNG (UnivariateFeatureSelector cho Regression)
selector = UnivariateFeatureSelector(
    featuresCol="scaled_features", outputCol="features", labelCol=target_col, selectionMode="percentile"
)
selector.setFeatureType("continuous").setLabelType("continuous").setSelectionThreshold(0.95) # Giữ 95% đặc trưng

# Gom Base Pipeline
base_pipeline = Pipeline(stages=[sql_trans] + indexers + encoders + [assembler, scaler, selector])

print("\n================ 3. TRAIN/TEST SPLIT & TRANSFORM BASE ================")
train_raw, test_raw = df_model.randomSplit([0.8, 0.2], seed=42)

# Transform 1 lần để tăng tốc độ GridSearch
base_model = base_pipeline.fit(train_raw)
train_data = base_model.transform(train_raw).cache()
test_data = base_model.transform(test_raw).cache()

print("\n================ 4. GRIDSEARCH & CROSS-VALIDATION (k=5) ================")
lr = LinearRegression(featuresCol="features", labelCol=target_col)
dt = DecisionTreeRegressor(featuresCol="features", labelCol=target_col)
rf = RandomForestRegressor(featuresCol="features", labelCol=target_col, seed=42)

paramGrid_lr = ParamGridBuilder().addGrid(lr.regParam, [0.01, 0.1]).addGrid(lr.elasticNetParam, [0.0, 0.5]).build()
paramGrid_dt = ParamGridBuilder().addGrid(dt.maxDepth, [5, 10]).addGrid(dt.maxBins, [32, 64]).build()
paramGrid_rf = ParamGridBuilder().addGrid(rf.numTrees, [50, 100]).addGrid(rf.maxDepth, [5, 10]).build()

models_to_train = {
    "Linear Regression": (lr, paramGrid_lr),
    "Decision Tree Regressor": (dt, paramGrid_dt),
    "Random Forest Regressor": (rf, paramGrid_rf)
}

evaluator_rmse = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")

results = []
best_overall_rmse = float('inf')
best_models_dict = {}
best_model_name = ""
best_predictions = None
best_overall_model = None

for name, (model, grid) in models_to_train.items():
    print(f"\n[+] Đang chạy Tuning cho {name}...")
    start_time = time.time()

    cv = CrossValidator(estimator=model, estimatorParamMaps=grid, evaluator=evaluator_rmse, numFolds=5, seed=42)
    cv_model = cv.fit(train_data)
    train_time = time.time() - start_time

    best_tuned_model = cv_model.bestModel
    best_models_dict[name] = best_tuned_model
    predictions = best_tuned_model.transform(test_data)

    rmse = evaluator_rmse.evaluate(predictions)
    mae = evaluator_mae.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)
    results.append({"Model": name, "RMSE": round(rmse, 4), "MAE": round(mae, 4), "R²": round(r2, 4), "Train time (s)": round(train_time, 2)})

    if rmse < best_overall_rmse:
        best_overall_rmse = rmse
        best_model_name = name
        best_overall_model = best_tuned_model
        best_predictions = predictions

print("\n================ 5. BẢNG KẾT QUẢ ĐÁNH GIÁ ================")
df_results = pd.DataFrame(results)
print(f"🌟 QUÁN QUÂN: {best_model_name}")
try:
    display(df_results)
except NameError:
    print(df_results.to_markdown(index=False))

print("\n================ 6. TẦM QUAN TRỌNG CỦA CÁC ĐẶC TRƯNG (TOP 10 FEATURE IMPORTANCE) ================")

if "Tree" in best_model_name or "Forest" in best_model_name:
    # 1. Lấy mảng giá trị quan trọng từ mô hình Quán quân
    importances = best_overall_model.featureImportances.toArray()

    # 2. Truy vết lấy danh sách Tên các đặc trưng từ Metadata của Pipeline
    # Lấy Metadata từ bước VectorAssembler (nằm ở vị trí stages[-3] trong base_model)
    attrs = train_data.schema["unscaled_features"].metadata["ml_attr"]["attrs"]

    full_feat_names = []
    # Lấy tên các cột số (numeric)
    if "numeric" in attrs:
        for attr in attrs["numeric"]: full_feat_names.append(attr["name"])
    # Lấy tên các cột đã One-Hot Encode (binary)
    if "binary" in attrs:
        for attr in attrs["binary"]: full_feat_names.append(attr["name"])

    # 3. Lọc lại các tên biến mà UnivariateFeatureSelector đã giữ lại
    # (Vì Selector đã loại bỏ bớt 5% features không quan trọng nhất)
    selected_indices = base_model.stages[-1].selectedFeatures
    final_names = [full_feat_names[i] for i in selected_indices]

    # 4. Tạo DataFrame Pandas để hiển thị đẹp mắt
    fi_df = pd.DataFrame({
        'Feature': final_names,
        'Importance': importances
    })

    # 5. Sắp xếp và lấy Top 10
    top10_fi = fi_df.sort_values(by="Importance", ascending=False).head(10)

    print(f"Bảng Top 10 đặc trưng ảnh hưởng nhất đến Phí vận chuyển ({best_model_name}):")
    print("-" * 65)
    print(top10_fi.to_string(index=False, justify='left', formatters={'Importance': '{:.6f}'.format}))
    print("-" * 65)
else:
    print(f"Mô hình {best_model_name} không hỗ trợ Feature Importance trực tiếp.")

print("\n================ 7. PHÂN TÍCH SAI SỐ (RESIDUAL ANALYSIS) ================")
# Phân tích sai lệch giống y hệt code gốc của bạn
sample_preds = best_predictions.select(target_col, "prediction").sample(fraction=0.1, seed=42).limit(5000).toPandas()
sample_preds['residual'] = sample_preds[target_col] - sample_preds['prediction']

mean_residual = sample_preds['residual'].mean()
std_residual = sample_preds['residual'].std()
skew_residual = sample_preds['residual'].skew()

print(f"Trung bình sai số (Mean Residual): {mean_residual:.4f}")
print(f"Độ lệch chuẩn sai số (Std Residual): {std_residual:.4f}")
print(f"Độ xiên (Skewness): {skew_residual:.4f}")

if abs(skew_residual) < 0.5:
    print("-> Kết luận: Phân phối sai số RẤT CHUẨN, đối xứng xung quanh mức 0 (Mô hình không bị thiên lệch).")
elif skew_residual > 0.5:
    print("-> Kết luận: Phân phối sai số LỆCH DƯƠNG (Mô hình thường có xu hướng dự đoán thấp hơn phí ship thực tế).")
else:
    print("-> Kết luận: Phân phối sai số LỆCH ÂM (Mô hình thường có xu hướng dự đoán cao hơn phí ship thực tế).")

# Vẽ biểu đồ Histogram
plt.figure(figsize=(10, 5))
sns.histplot(sample_preds['residual'], kde=True, bins=50, color='blue')
plt.title(f'Biểu đồ phân phối sai số (Residuals) Phí vận chuyển - {best_model_name}')
plt.xlabel('Sai số (Thực tế - Dự đoán)')
plt.ylabel('Tần suất')
plt.axvline(0, color='red', linestyle='--')
plt.show()

print("\n================ 8. LƯU END-TO-END PIPELINE CHO GRADIO ================")
# Gắn liền Base Pipeline và Best Model lại để sẵn sàng deploy
final_freight_pipeline = PipelineModel(stages=[base_model, best_models_dict[best_model_name]])

model_save_path = "/content/gdrive/MyDrive/BIGDATA/gradio_freight_pipeline_model"
final_freight_pipeline.write().overwrite().save(model_save_path)
print(f"✅ Đã lưu End-to-End Pipeline thành công tại: {model_save_path}")